<a href="https://colab.research.google.com/github/krishna6677888/krishna-portfolio/blob/main/Hr_Based_chat_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building an HR Chatbot for Leave Policies

To build an HR chatbot that can answer questions about leave policies, we'll use a technique called Retrieval Augmented Generation (RAG). This involves:

1.  **Extracting Documents**: Unzip the provided archive to access the policy documents.
2.  **Document Processing**: Load, split, and embed the policy documents.
3.  **Vector Store Creation**: Store the embedded documents in a vector database.
4.  **RAG Pipeline**: Set up a system to retrieve relevant document snippets based on a user's query and then use a Large Language Model (LLM) to generate a concise answer.

In [1]:
# Unzip the provided file to access the policy documents.
import zipfile
import os

zip_file_path = '/content/niat-masterclass-rag-challenge (1).zip'
extraction_path = './hr_policy_documents'

# Create the directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"Files extracted to: {extraction_path}")

# List the extracted files to see what we have
for root, dirs, files in os.walk(extraction_path):
    for file in files:
        print(os.path.join(root, file))

Files extracted to: ./hr_policy_documents
./hr_policy_documents/zyro-dynamics-hr-corpus/00_Company_Profile.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/08_Prevention_of_Sexual_Harassment_Policy.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/sample_submission.xlsx
./hr_policy_documents/zyro-dynamics-hr-corpus/05_Performance_Review_Policy.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/02_Leave_Policy.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/01_Employee_Handbook.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/09_Onboarding_and_Separation_Policy.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/03_Work_From_Home_Policy.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/06_Compensation_and_Benefits_Policy.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/Starter_Notebook.ipynb
./hr_policy_documents/zyro-dynamics-hr-corpus/04_Code_of_Conduct.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/10_Travel_and_Expense_Policy.pdf
./hr_policy_documents/zyro-dynamics-hr-corpus/07

Now that the files are extracted, the next step is to explore the content of these files to identify the relevant policy documents (e.g., PDF, TXT, DOCX files) and then proceed with loading and processing them for our chatbot.

In [2]:
# Install necessary libraries
!pip install pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.9 MB/s eta 0:00:00


In [3]:
from pypdf import PdfReader
import os

# Define the directory where the documents were extracted
documents_dir = './hr_policy_documents/zyro-dynamics-hr-corpus'

# List all PDF files in the directory
pdf_files = [os.path.join(documents_dir, f) for f in os.listdir(documents_dir) if f.endswith('.pdf')]

# Create a list to store the document texts
documents_content = []

for pdf_file in pdf_files:
    try:
        reader = PdfReader(pdf_file)
        text = ''
        for page in reader.pages:
            text += page.extract_text()
        documents_content.append({
            'file_name': os.path.basename(pdf_file),
            'content': text
        })
        print(f"Successfully loaded: {os.path.basename(pdf_file)}")
    except Exception as e:
        print(f"Error loading {os.path.basename(pdf_file)}: {e}")

print(f"\nTotal documents loaded: {len(documents_content)}")

# Display a sample of the content from the 'Leave Policy' if available
leave_policy_content = next((doc['content'] for doc in documents_content if 'Leave_Policy.pdf' in doc['file_name']), None)

if leave_policy_content:
    print("\n--- Sample from Leave Policy Document ---")
    print(leave_policy_content[:1000]) # Print first 1000 characters
else:
    print("\nLeave Policy document not found or could not be loaded.")

Successfully loaded: 00_Company_Profile.pdf
Successfully loaded: 08_Prevention_of_Sexual_Harassment_Policy.pdf
Successfully loaded: 05_Performance_Review_Policy.pdf
Successfully loaded: 02_Leave_Policy.pdf
Successfully loaded: 01_Employee_Handbook.pdf
Successfully loaded: 09_Onboarding_and_Separation_Policy.pdf
Successfully loaded: 03_Work_From_Home_Policy.pdf
Successfully loaded: 06_Compensation_and_Benefits_Policy.pdf
Successfully loaded: 04_Code_of_Conduct.pdf
Successfully loaded: 10_Travel_and_Expense_Policy.pdf
Successfully loaded: 07_IT_and_Data_Security_Policy.pdf

Total documents loaded: 11

--- Sample from Leave Policy Document ---
Zyro Dynamics Pvt. Ltd.
Leave Policy
Doc Code: ZDL-HR-002
Confidential — For Internal Use Only
Page 1
 Zyro Dynamics Pvt. Ltd.
 
 Navigate the Future
 Leave Policy
 Document Code
 ZDL-HR-002
 Version
 V.04
 Effective Date
 01 April 2025
 Document Owner
 HR Department
INTRODUCTION
We value the importance of finding a healthy balance between your prof

Now that we've loaded the content of the PDF documents, including the 'Leave Policy', the next step is to process this raw text. This will involve splitting the text into smaller, manageable chunks and then embedding these chunks to prepare them for the vector store. This process is crucial for effective retrieval in our RAG system.

In [17]:
# Install langchain and langchain-community for text splitting, ensuring upgrade and verbose output
!pip install --upgrade langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.9
    Uninstalling langchain-1.3.9:
      Successfully uninstalled langchain-1.3.9


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, # Max size of each chunk
    chunk_overlap  = 100, # Overlap between chunks to maintain context
    length_function = len,
    is_separator_regex = False,
)

# Prepare all document content into a single list of strings
all_texts = [doc['content'] for doc in documents_content]

# Split the documents into chunks
# The splitter expects a list of strings, so we pass all_texts
chunks = text_splitter.create_documents(all_texts)

print(f"Total number of chunks created: {len(chunks)}")

# Display a sample chunk
if chunks:
    print("\n--- Sample Chunk ---")
    print(chunks[0].page_content[:500]) # Print first 500 characters of the first chunk
else:
    print("No chunks were created.")

Total number of chunks created: 77

--- Sample Chunk ---
Zyro Dynamics Pvt. Ltd.
Company Profile
Doc Code: ZDL-CORP-001
Confidential — For Internal Use Only
Page 1
 Zyro Dynamics Pvt. Ltd.
 
 Navigate the Future
 Company Profile
 Document Code
 ZDL-CORP-001
 Version
 V.01
 Effective Date
 01 April 2025
 Document Owner
 Corporate Communications
COMPANY OVERVIEW
Zyro Dynamics Pvt. Ltd. is a product-based enterprise software company headquartered in the United States. 
Founded in 2014 by Lamine Yamal and Aitana Bonmati, the company was built on a
single 


With the documents now split into smaller chunks, the next critical step is to convert these text chunks into numerical representations called embeddings. These embeddings capture the semantic meaning of the text, allowing us to perform similarity searches. We'll then store these embeddings in a vector database, which will enable our chatbot to quickly find the most relevant policy information for any given query.

In [6]:
# Install necessary libraries for embeddings and vector store
!pip install sentence-transformers chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [10]:
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize the embedding model
# We'll use a local sentence transformer model for creating embeddings
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the vector store using Chroma
# We'll persist the database to disk for later use
persist_directory = "./hr_policy_chroma_db"
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_function,
    persist_directory=persist_directory
)

# Persist the database to disk
vectordb.persist()
print(f"Vector database created with {len(chunks)} embeddings and persisted to '{persist_directory}'.")

# Verify by loading the collection
loaded_vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)
print(f"Number of items in the loaded vector database: {loaded_vectordb._collection.count()}")

/tmp/ipykernel_6048/2600000439.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import SentenceTransformerEmbeddings
/tmp/ipykernel_6048/2600000439.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created with 77 embeddings and persisted to './hr_policy_chroma_db'.
Number of items in the loaded vector database: 77


/tmp/ipykernel_6048/2600000439.py:18: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()
/tmp/ipykernel_6048/2600000439.py:22: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  loaded_vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)


Excellent! We now have our policy documents processed, chunked, embedded, and stored in a Chroma vector database. This is the core of our RAG system. The final step is to set up a retrieval chain that can take a user's question, find the most relevant chunks from our database, and then use a Large Language Model (LLM) to generate a coherent answer based on that information. We'll use a generative model from the `google.generativeai` library for this.

In [11]:
# Install google-generativeai and dotenv for API key management
!pip install -U --quiet google-generativeai python-dotenv

In [12]:
import google.generativeai as genai
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Configure the generative AI model with your API key
genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))

# Initialize the Generative Model (e.g., Gemini Pro)
# You can choose a different model if needed, like 'gemini-pro'
llm = genai.GenerativeModel('gemini-pro')

print("Generative model initialized.")

Generative model initialized.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


With the generative model initialized, we can now set up the retrieval component. This will involve creating a retriever from our `vectordb` that can fetch the most relevant document chunks based on a user's query. Finally, we'll combine the retriever and the LLM into a powerful RAG chain using LangChain, ready to answer questions about the HR policies.

In [22]:
from langchain.chains.retrieval import RetrievalQA
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

# Re-initialize the vector database to ensure it's properly loaded
# This is necessary if the kernel restarts or the database was just created/persisted
persist_directory = "./hr_policy_chroma_db"

# Ensure embedding_function is defined (from previous steps)
# If running this cell independently, uncomment the following line:
# from langchain_community.embeddings import SentenceTransformerEmbeddings
# embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)

# Create a retriever from the vector database
retriever = vectordb.as_retriever(search_kwargs={"k": 3}) # Retrieve top 3 relevant chunks

# Initialize the LLM for the RAG chain
# Using the ChatGoogleGenerativeAI wrapper for LangChain compatibility
rag_llm = ChatGoogleGenerativeAI(model="gemini-pro")

# Define a custom prompt template for our HR chatbot
# This helps guide the LLM to provide relevant and concise answers
prompt_template = """You are an HR assistant chatbot. Use the following pieces of context to answer the user's question. If you don't know the answer, just say that you don't know, don't try to make up an answer. Keep the answer concise and relevant to HR policies.

Context: {context}
Question: {question}

Helpful Answer:"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(prompt_template)

# Create the RAG chain
qa_chain = RetrievalQA.from_chain_type(
    rag_llm,
    chain_type="stuff", # 'stuff' combines all retrieved documents into one prompt
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

print("RAG chain setup complete.")

ModuleNotFoundError: No module named 'langchain.chains'

In [21]:
print('--- Verifying langchain package installations ---')
!pip list | grep langchain

--- Verifying langchain package installations ---
langchain                                1.3.11
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.4.7
langchain-protocol                       0.0.17
langchain-text-splitters                 1.1.2


Our HR chatbot is now ready! You can now start asking questions related to the HR policies we loaded. For example, you can ask about leave types, work-from-home policies, or performance review processes. Let's try an example query.

You can continue asking more questions to test the chatbot's capabilities. If you encounter any issues or want to refine the answers, we can adjust parameters like `chunk_size`, `chunk_overlap`, the number of retrieved documents (`k`), or even the prompt template.